# ChadGPT — open, run, and test

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://huggingface.co/sam-eer12/chadGPT/colab)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://huggingface.co/sam-eer12/chadGPT/kaggle)

This notebook installs the required packages and automatically fetches the inference code, tokenizer, configuration, Gradio app, and model weights from **sam-eer12/chadGPT** on Hugging Face. No code to clone, files to upload, or Hugging Face login/token is needed.

1. Open either launcher link and sign in to Colab or Kaggle.
2. On **Kaggle**, enable **Internet** in notebook settings. Optionally select a GPU accelerator; CPU also works. Complete any account verification that Kaggle requests. On **Colab**, connect a runtime; a GPU is optional.
3. Run **Install dependencies → Download and load → Test inference** in order. Read or change the example prompt in the test cell to try the model inside the notebook.
4. Optionally run **Launch and share** to create a public **`https://…gradio.live`** chat link. Keep its cell and runtime running while sharing it.

The first model download is approximately **1 GB**. CUDA is selected when available, otherwise Apple MPS or CPU. Installation keeps the notebook's existing PyTorch/CUDA stack. No Hugging Face PRO subscription is required.

**Colab:** use the inference test directly in the notebook. [Free managed Colab runtimes restrict using a web UI as the primary interface](https://research.google.com/colaboratory/faq.html) and may terminate shared Gradio demos. Run the optional sharing cell only in a runtime whose rules allow it.

[Model card and full training history](https://huggingface.co/sam-eer12/chadGPT) · [Gradio app source](https://huggingface.co/sam-eer12/chadGPT/blob/main/playground/app.py)

ChadGPT is an experimental 250M-parameter model trained from scratch on Kaggle over approximately two months. Its answers and generated code can be incorrect.

## 1. Install inference dependencies

This cell deliberately does not install the Space-only CPU PyTorch wheel. Kaggle and Colab normally provide PyTorch already. On a local notebook without PyTorch, install it separately first.

In [ ]:
%pip install -q gradio==6.29.1 tiktoken==0.12.0 safetensors==0.8.0 huggingface_hub==1.21.0

## 2. Download the existing app and load the model

The Gradio interface is reused directly from the public Hugging Face repository. The app pins its weights and inference implementation to the verified initial model release.

In [ ]:
import importlib.util
import os
import sys

import torch
from huggingface_hub import hf_hub_download

os.environ['CHADGPT_DEVICE'] = 'auto'
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'

# Close only this notebook's previous demo if this cell is rerun.
if 'playground_app' in globals():
    playground_app.demo.close()

app_path = hf_hub_download('sam-eer12/chadGPT', 'playground/app.py')
spec = importlib.util.spec_from_file_location('chadgpt_playground', app_path)
playground_app = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = playground_app
spec.loader.exec_module(playground_app)

print('Loading ChadGPT (approximately 1 GB on first download)…')
model, tokenizer, _ = playground_app.get_runtime()
print('Model ready on:', next(model.parameters()).device)
print('PyTorch:', torch.__version__)
print('Run the next cell to test inference; Gradio sharing is optional.')

## 3. Test inference inside the notebook

Run this cell for a sample response. Change **prompt** and run it again to ask another question. This test prints the answer here and does not start a public web server.

In [ ]:
prompt = "Explain what a neural network is in simple terms." #@param {type:"string"}
max_new_tokens = 96 #@param {type:"integer"}

model, tokenizer, chat = playground_app.get_runtime()
response = chat(
    model, tokenizer,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    max_new_tokens=int(max_new_tokens), temperature=0.7, top_k=40,
)
print(response or "The model ended its response without generating text. Try another prompt.")

## 4. Optional: launch and share Gradio

Run the cell below. Copy the **Running on public URL: `https://…gradio.live`** link from its output. Anyone with that link can open the chat interface.

**Keep this cell running and the runtime connected.** The cell remains busy intentionally. The URL stops working when the runtime or demo stops; share links also expire after at most one week. It is a temporary demo link, not permanent Hugging Face hosting.

[How Gradio share links work](https://gradio.app/guides/sharing-your-app)

In [ ]:
playground_app.demo.launch(share=True, debug=True, inline=True, show_error=True)

## 5. Stop the demo when finished

First interrupt the running launch cell using its stop button, then run this optional cleanup cell. You can also stop/disconnect the notebook runtime.

In [ ]:
if 'playground_app' in globals():
    playground_app.demo.close()
    print('Public demo stopped.')